# 02 — Train Classifier (M3): PhoBERT phân loại gói thầu

Input: `data/processed/classifier_dataset.jsonl`. Nhãn: hang_hoa | xay_lap | tu_van | phi_tu_van | hon_hop.
Output: checkpoint tại `models/classifier_phobert/`.
Metric: macro-F1, confusion matrix. Baseline bắt buộc: TF-IDF + LogisticRegression.

In [ ]:
!git clone https://github.com/dannd/autotender-vn.git /content/autotender-vn 2>/dev/null || echo 'Repo đã tồn tại (chạy lại notebook) hoặc private — kiểm tra quyền truy cập.'
%cd /content/autotender-vn

In [ ]:
# data/processed/*.jsonl KHÔNG nằm trong git (xem .gitignore) — phải sinh lại từ
# data/samples/tender_notices.jsonl (đã commit) bằng script này.
!python scripts/build_dataset.py

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
import json
from pathlib import Path

DATA_PATH = Path('/content/autotender-vn/data/processed/classifier_dataset.jsonl')
records = [json.loads(l) for l in open(DATA_PATH, encoding='utf-8')]
labels = sorted({r['label'] for r in records})
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print(len(records), 'records —', labels)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

texts = [r['text'] for r in records]
y = [r['label'] for r in records]
X_train, X_val, y_train, y_val = train_test_split(texts, y, test_size=0.2, random_state=42, stratify=y if len(set(y)) > 1 else None)

vec = TfidfVectorizer(max_features=5000)
Xt = vec.fit_transform(X_train)
Xv = vec.transform(X_val)
baseline = LogisticRegression(max_iter=1000).fit(Xt, y_train)
print('--- Baseline TF-IDF + LogisticRegression ---')
print(classification_report(y_val, baseline.predict(Xv)))

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(texts_, labels_):
    return Dataset.from_dict({'text': texts_, 'label': [label2id[l] for l in labels_]})

train_ds = to_hf(X_train, y_train).map(lambda b: tokenizer(b['text'], truncation=True, max_length=256), batched=True)
val_ds = to_hf(X_val, y_val).map(lambda b: tokenizer(b['text'], truncation=True, max_length=256), batched=True)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id)
collator = DataCollatorWithPadding(tokenizer)

def compute_metrics(eval_pred):
    preds, labels_ = eval_pred
    preds = np.argmax(preds, axis=1)
    return {'macro_f1': f1_score(labels_, preds, average='macro')}

args = TrainingArguments(
    output_dir='/content/clf_out', num_train_epochs=10, per_device_train_batch_size=8,
    per_device_eval_batch_size=8, eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='macro_f1', logging_steps=5, report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collator, compute_metrics=compute_metrics)

In [ ]:
trainer.train()
trainer.save_model('/content/models/classifier_phobert')
tokenizer.save_pretrained('/content/models/classifier_phobert')
print('Checkpoint saved — tải /content/models/classifier_phobert về models/classifier_phobert/ trong repo local.')

In [ ]:
# Nen checkpoint thanh zip va tai truc tiep ve may (khong can mount Drive)
from google.colab import files
import shutil
shutil.make_archive('classifier_phobert', 'zip', '/content/models/classifier_phobert')
files.download('classifier_phobert.zip')
print('Giai nen classifier_phobert.zip vao thu muc models/classifier_phobert/ trong repo local.')